# Lecture 11 — Randomness, Iteration, and Simulation

**Introduction to Data Science · 2026 Spring · Pusan National University · School of CSE**
**Donghee Choi** · `dchoi@pusan.ac.kr`

This notebook contains every code example from slide deck `11_randomness_simulation.pdf`. Click the *Open in Colab* badge in the topic page to run it directly.

## Learning Goals
1. **Chance** — basic rules: range, complement, equally-likely outcomes, multiplication rule.
2. **Randomness** — generate random samples with `np.random.choice` and count outcomes with Boolean comparisons.
3. **Iteration** — repeat an experiment with a `for` loop and accumulate results.
4. **Simulation** — one trial gives one statistic; many trials give an *empirical distribution*.
5. **Case study** — solve the Monty Hall problem by simulation.

**Prerequisites**: Lecture 7 (Python/NumPy), 8 (Pandas), 10 (Sampling).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---
## Part A — Chance and Randomness

### A.0a Chance: basics

- **Chance** is the possibility of something happening.
- Lowest value: `0` — chance of an event that is impossible.
- Highest value: `1` (or `100%`) — chance of an event that is certain.
- **Complement.** If an event has chance 70%, the chance that it does *not* happen is
  - `100% - 70% = 30%`
  - `1 - 0.7 = 0.3`
  - i.e., `P(~A) = 1 - P(A)`.

### A.0b Equally-likely outcomes

If all outcomes are equally likely, the chance of an event `A` is

`P(A) = (number of outcomes that make A happen) / (total number of outcomes)`

**Example.** A fair die shows an even number:
`P(A) = #{2, 4, 6} / #{1, 2, 3, 4, 5, 6} = 3 / 6 = 0.5`.

In [ ]:
faces = np.arange(1, 7)
even = (faces % 2 == 0)
print('outcomes:', faces)
print('A = even:', even)
print('P(A):', np.count_nonzero(even) / len(faces))   # 0.5

### A.0c Multiplication Rule

For two events A and B,

`P(A and B) = P(A) * P(B given that A has happened)`

The product is at most each of the two factors — the more conditions an event has to satisfy, the less likely it is.

**Example.** Three cards: ace of hearts, king of diamonds, queen of spades. Shuffle and draw two cards *without replacement*. What is `P(Queen first, then King)`?

- `P(Queen on draw 1) = 1/3`
- Given the queen is gone, `P(King on draw 2) = 1/2`
- `P(Q, then K) = 1/3 * 1/2 = 1/6`.

In [ ]:
# Verify by simulation
deck = np.array(['Ace', 'King', 'Queen'])

def queen_then_king():
    draw = np.random.choice(deck, 2, replace=False)
    return (draw[0] == 'Queen') and (draw[1] == 'King')

trials = 100_000
hits = sum(queen_then_king() for _ in range(trials))
print('empirical P(Q, then K):', hits / trials)   # ~ 0.1667

### A.0d Addition Rule

If event `A` can happen in **exactly one of two (mutually exclusive) ways**,

`P(A) = P(first way) + P(second way)`

The total is at least as large as either individual way.

**Example.** With the same three cards, what is `P(one drawn card is King and the other is Queen)`?

- Way 1: King then Queen — `1/3 * 1/2 = 1/6`
- Way 2: Queen then King — `1/3 * 1/2 = 1/6`
- Total: `1/6 + 1/6 = 2/6 = 1/3`.

In [ ]:
def king_and_queen():
    draw = np.random.choice(deck, 2, replace=False)
    return ('King' in draw) and ('Queen' in draw)

trials = 100_000
hits = sum(king_and_queen() for _ in range(trials))
print('empirical P(King and Queen, any order):', hits / trials)   # ~ 0.333

### A.0e Complement: "At Least One Head"

`P(at least one head)` = `1 - P(no heads at all)`.

In `N` tosses, "no heads" means every toss is Tails:

`P(all Tails) = (1/2)^N`

So `P(at least one head) = 1 - (1/2)^N`.

| N | `1 - (1/2)^N` |
|---|---|
| 1 | 0.5 |
| 3 | 0.875 (87.5%) |
| 10 | ≈ 0.999 (99.9%) |

In [ ]:
for N in [1, 3, 10, 20]:
    p = 1 - (1/2)**N
    print(f'N={N:2d}   P(at least one H) = {p:.6f}')

### A.0f Problem-solving method

A simple decision tree for chance problems:

1. **Ask what event must happen on the first trial.**
2. If there is a *clear* answer with a known probability (e.g. "not a six") → use the **multiplication rule** to chain it with later events.
3. If there is *no* single clear answer (e.g. "could be K or Q, but then the next one would have to be the other") → list all the distinct ways the event could occur and **add up their chances**.
4. If that list is long and messy → look at the **complement**. The complement of "at least one X" is "no X at all", which is often a simple multiplication.

### A.0g Discussion — Rick and Morty

A population of 100 people includes **Rick** and **Morty**. Sample two people at random *without replacement*.

**P(both Rick and Morty are in the sample)**

Two ways:
- First Rick, then Morty: `(1/100) * (1/99)`
- First Morty, then Rick: `(1/100) * (1/99)`

`P = (1/100)*(1/99) + (1/100)*(1/99) ≈ 0.0002`

**P(neither Rick nor Morty is in the sample)**

`P = (98/100) * (97/99) ≈ 0.9602`.

In [ ]:
p_both = 2 * (1/100) * (1/99)
p_neither = (98/100) * (97/99)
p_exactly_one = 1 - p_both - p_neither

print(f'P(both)        = {p_both:.6f}')
print(f'P(neither)     = {p_neither:.6f}')
print(f'P(exactly one) = {p_exactly_one:.6f}')
print(f'sum            = {p_both + p_neither + p_exactly_one:.6f}')   # 1.0

### A.1 Comparison Expressions — every comparison returns a `bool`

`>`, `<`, `==`, `!=`, `>=`, `<=` always return `True` or `False`.

In [ ]:
x = 2; y = 3
print(x > 1)        # True
print(x > y)        # False
print(y >= 3)       # True
print(x == y)       # False
print(x != 2)       # False
print(2 < x < 5)    # False  (chained: 2 < x AND x < 5)

**Strings compare too** — alphabetical order first, then length when prefix is the same.

In [ ]:
print('Dog' > 'Catastrophe' > 'Cat')   # True
print('a' < 'b')                       # True   (alphabetical)
print('abc' < 'abcd')                  # True   (shorter < longer with shared prefix)

### A.2 Aggregating Comparisons — sum of bools = count of `True`

Since `True == 1` and `False == 0`, `sum` on a list/array of bools gives the count of `True`.

In [ ]:
print(1 + 0 + 1)                  # 2
print(True + False + True)        # 2  (True==1, False==0)

print(sum([1, 0, 1]))             # 2
print(sum([True, False, True]))   # 2

### A.3 Comparing an Array and a Value

- **Comparison applies to each element** of the array → result is a Boolean array.
- **The result array can be aggregated** the same way as the scalar case.

In [ ]:
tosses = np.array(['Tails', 'Heads', 'Tails', 'Heads', 'Heads'])
tosses == 'Tails'
# array([ True, False,  True, False, False])

In [ ]:
np.count_nonzero(tosses == 'Tails')   # 2

In [ ]:
sum(tosses == 'Tails')                # 2   (True == 1)

### A.4 `np.random.choice` — drawing one outcome

Put the possible outcomes in an array, then call `np.random.choice` to pick one uniformly at random. Re-run the cell — the result changes each time.

In [ ]:
two_groups = np.array(['treatment', 'control'])
np.random.choice(two_groups)

### A.5 `np.random.choice` — drawing many outcomes

Pass an integer `N` as the second argument to draw `N` outcomes at once. Result is a numpy array.

In [ ]:
np.random.choice(two_groups, 10)

### A.6 Sampling **With** vs **Without** Replacement

- **With replacement** (default, `replace=True`) — each draw is **independent**; the same item can appear multiple times. Coin tosses, dice rolls, repeated polling.
- **Without replacement** (`replace=False`) — draws are **linked**, each item appears at most once. Dealing cards, picking a committee, sampling distinct people. Maximum sample size = array length.

In [ ]:
deck = np.array(['A', 'B', 'C', 'D', 'E'])

# WITH replacement — same item can repeat
np.random.choice(deck, 4)
# e.g. array(['C', 'A', 'C', 'B'])

In [ ]:
# WITHOUT replacement — all distinct
np.random.choice(deck, 4, replace=False)
# e.g. array(['D', 'A', 'E', 'B'])

In [ ]:
# replace=False cannot draw more than the array length (this raises ValueError):
# np.random.choice(deck, 6, replace=False)

---
## Part B — Iteration

### B.1 `for` over `np.arange`

In [ ]:
for i in np.arange(5):
    print('iteration', i)

### B.2 Augmenting arrays — `np.append`

`np.append` does not modify the original array — it returns a *new* array with the addition.

- `np.append(array_1, value)` — new array with `value` appended to `array_1`. `value` must be of the same type as the elements of `array_1`.
- `np.append(array_1, array_2)` — new array with `array_2` appended to `array_1`. Element types must be compatible.

In [ ]:
a = np.array([1, 2, 3])

# append a single value
np.append(a, 4)
# array([1, 2, 3, 4])

In [ ]:
# append another array
np.append(a, np.array([10, 20]))
# array([ 1,  2,  3, 10, 20])

In [ ]:
# the original is NOT modified
a
# array([1, 2, 3])

### B.3 Accumulating in a loop

Pattern: empty array → loop → append each new outcome → final array.

In [ ]:
outcomes = np.array([])
for i in np.arange(5):
    pick = np.random.choice(['H', 'T'])
    outcomes = np.append(outcomes, pick)
outcomes

### B.4 Example — a dice bet

**Rules.** Roll one fair die. 1 or 2 → lose 1. 3 or 4 → break even. 5 or 6 → win 1.

In [ ]:
def one_bet(x):
    if x <= 2:
        return -1
    elif x <= 4:
        return 0
    else:
        return 1

def bet_on_one_roll():
    x = np.random.choice(np.arange(1, 7))
    return one_bet(x)

bet_on_one_roll()

### B.5 Repeat 300 times and look at the distribution

In [ ]:
outcomes = np.array([])
for i in np.arange(300):
    outcomes = np.append(outcomes, bet_on_one_roll())

table = pd.DataFrame({'Outcome': outcomes})
counts = (table.groupby('Outcome')['Outcome']
               .count()
               .reset_index(name='count'))
counts

In [ ]:
ax = counts.plot.barh(x='Outcome', y='count', legend=False)
ax.set_xlabel('count'); ax.set_title('300 dice bets')
plt.show()

Each of `-1`, `0`, `+1` should appear roughly 1/3 of the time — the empirical distribution of the bet.

---
## Part C — Simulation

When one trial returns a *number*, repeating the trial many times lets us see the distribution of that number.

### C.1 One simulated value

**Experiment**: flip a coin 100 times and count the Heads.

In [ ]:
coin = np.array(['Heads', 'Tails'])

def one_simulated_value():
    flips = np.random.choice(coin, 100)
    return np.count_nonzero(flips == 'Heads')

one_simulated_value()

### C.2 Repeat 20,000 times to build the distribution

In [ ]:
num_repetitions = 20000

heads = np.array([])
for i in np.arange(num_repetitions):
    heads = np.append(heads, one_simulated_value())

len(heads)

In [ ]:
rst = pd.DataFrame({'Number of Heads': heads})
ax = rst['Number of Heads'].plot.hist(bins=np.arange(30, 71), edgecolor='white')
ax.set_xlabel('Number of Heads in 100 flips')
ax.set_title(f'Empirical distribution over {num_repetitions} repetitions')
plt.show()

The histogram is centered near 50 and bell-shaped — a preview of the Normal distribution.

### C.3 Aside — probability of at least one Head in `N` tosses

Sometimes algebra is easier than simulation. Pick whichever is clearer.

In [ ]:
tosses = np.arange(1, 51)
results = pd.DataFrame({
    'Tosses': tosses,
    'Chance of at least one H': 1 - (1/2)**tosses
})
ax = results.plot.scatter(x='Tosses', y='Chance of at least one H')
plt.show()

---
## Part D — Monty Hall Problem

Three doors. One hides a **car**, two hide **goats**. You pick a door. The host (who knows where the car is) opens a *different* door that has a goat behind it, and offers you a chance to switch. Should you switch?

<img src="https://upload.wikimedia.org/wikipedia/commons/3/3f/Monty_open_door.svg" alt="Monty Hall — host opens a goat door" width="380">

<sub>Image: *Monty open door* (Wikipedia, public domain). The host opens one of the two unchosen doors revealing a goat — never the door with the car.</sub>

### Why switch wins 2/3 — the easy explanation

Initially every door is equally likely (`1/3`). The contestant's first pick has chance `1/3` of being the car, so the **other two doors together have chance `2/3`**.

The host then opens one of those two — but always a *goat*. He gives you no information about your own door (it stays at `1/3`), and he concentrates the entire remaining `2/3` onto the **single unopened door that wasn't yours**. Switching takes that `2/3`.

| Outcome of first pick | Chance | If you keep | If you switch |
|---|---|---|---|
| You picked the **car** (1 door) | 1/3 | win | lose |
| You picked a **goat** (2 doors) | 2/3 | lose | **win** |

So switching wins with probability **2/3**, keeping wins with probability **1/3**.

<img src="https://upload.wikimedia.org/wikipedia/commons/0/05/Monty_tree_door1.svg" alt="Monty Hall — tree diagram if you initially pick door 1" width="450">

<sub>Image: *Monty tree door1* (Wikipedia, public domain). All possible outcomes when the contestant initially picks door 1, summing to the 1/3 vs 2/3 split.</sub>

In [ ]:
goats = np.array(['first goat', 'second goat'])
hidden = np.append(goats, 'car')

def other_goat(x):
    if x == 'first goat':
        return 'second goat'
    if x == 'second goat':
        return 'first goat'

def monty_hall_game():
    contestant = np.random.choice(hidden)
    if contestant == 'car':
        revealed = np.random.choice(goats)
        remaining = other_goat(revealed)
    elif contestant == 'first goat':
        revealed = 'second goat'
        remaining = 'car'
    else:
        revealed = 'first goat'
        remaining = 'car'
    return [contestant, revealed, remaining]

monty_hall_game()

### D.1 Simulate 10,000 games

In [ ]:
games = pd.DataFrame(columns=['Guess', 'Revealed', 'Remaining'])
for i in np.arange(10000):
    games.loc[i] = monty_hall_game()

games.head()

In [ ]:
# Distribution of the original guess (= outcome if you keep)
original = (games.groupby('Guess')['Guess']
                  .count()
                  .reset_index(name='if_kept'))

# Distribution of the remaining door (= outcome if you switch)
remaining = (games.groupby('Remaining')['Remaining']
                  .count()
                  .reset_index(name='if_switched'))

joined = original.merge(remaining, left_on='Guess', right_on='Remaining')
joined

In [ ]:
ax = joined.set_index('Guess')[['if_kept', 'if_switched']].plot.barh()
ax.set_xlabel('count'); ax.set_title('Monty Hall — keep vs switch (10,000 games)')
plt.show()

`car` appears as the *remaining door* about 67% of the time — so **switching wins ~2/3 of the games**.

---
## Simulation Recipe

1. **Define the experiment** — a function that runs one trial and returns the statistic.
2. **Repeat** — `for i in np.arange(N)`.
3. **Accumulate** — append to an empty array.
4. **Visualize** — `plot.hist` or `plot.barh`.
5. **Read off** the empirical distribution.

Next lecture: apply the same pattern to data instead of dice — Estimation, Bootstrap, Confidence Intervals.